# Galaxy Morphology Classification & Clustering
## Data Mining Project — SDSS DR7 Morphological Catalogue (Barchi et al. 2019)

**Science Question:** Can non-parametric morphological features derived from photometric imaging reliably classify galaxies as elliptical or spiral, and do unsupervised clustering methods reveal substructure consistent with known morphological classes?

**Dataset:** Barchi et al. (2019) — Machine and Deep Learning morphological catalogue of 670,560 galaxies from the Sloan Digital Sky Survey Data Release 7 (SDSS-DR7).

**Techniques Used:**
- Supervised Classification: Decision Tree, Random Forest, Logistic Regression
- Unsupervised Clustering: K-Means with Elbow Method & Silhouette Analysis
- Dimensionality Reduction: PCA (for cluster visualisation)

---
**Notebook structure:**
1. Setup & Imports
2. Data Loading & Inspection
3. Data Cleaning & Pre-processing
4. Exploratory Data Analysis (EDA)
5. Classification — Model Training & Evaluation
6. Clustering — K-Means
7. Summary of Findings

## 1. Setup & Imports

In [ ]:
# Standard scientific Python stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: classification models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.preprocessing import StandardScaler

# Scikit-learn: clustering & dimensionality reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Reproducibility seed
RANDOM_STATE = 42

# ── Colour palette ──────────────────────────────────────────────────────────
# Unique violet / teal / coral scheme used consistently across all figures
C1 = '#7B2D8B'   # deep violet  — Elliptical / primary model bars
C2 = '#00C49A'   # teal green   — Spiral / secondary model bars
C3 = '#FF6B6B'   # coral red    — third cluster / accent
CMAP_SEQ = 'BuPu'
CMAP_DIV = 'PRGn'
BG = '#F7F4FA'   # soft lavender background

# Apply global matplotlib style
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': '#4A3560', 'axes.labelcolor': '#2D1B4E',
    'xtick.color': '#2D1B4E', 'ytick.color': '#2D1B4E',
    'text.color': '#2D1B4E', 'grid.color': '#D4C8E8',
})

print('All imports successful. Colour palette loaded.')

## 2. Data Loading & Inspection

The dataset contains 670,560 galaxies with 20 columns including morphological non-parametric parameters (C, A, S, G2, H), a size metric (K), error flags, and classification labels from both a traditional ML model and a CNN.

In [ ]:
# Load the full dataset — set low_memory=False to avoid mixed-type warnings
df = pd.read_csv('Barchi19_Morph-catalog_670k-galaxies.csv', low_memory=False)

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn names:')
print(df.columns.tolist())

In [ ]:
# Preview the first few rows to understand structure
df.head()

In [ ]:
# Inspect data types and check for any immediately obvious issues
df.info()

In [ ]:
# Descriptive statistics for the morphological feature columns
FEATURES = ['C', 'A', 'S', 'G2', 'H', 'K']
df[FEATURES].describe()

In [ ]:
# Inspect the ML2classes label column — it contains numeric labels AND special strings
print('ML2classes value counts:')
print(df['ML2classes'].value_counts())
# 0 = elliptical, 1 = spiral, 'U' = uncertain, '--' = unclassified by the Decision Tree

In [ ]:
# Check the Error flag distribution
# Error=0 means successful computation; non-zero values indicate specific measurement failures
print('Error flag distribution:')
print(df['Error'].value_counts())
# Per the README: Error=2 means Rp (Petrosian radius) could not be calculated

## 3. Data Cleaning & Pre-processing

Three types of data quality issues must be addressed before analysis:

1. **Sentinel values (-9999.x):** Used by the CyMorph pipeline to flag features that could not be computed. These are not real measurements and will severely distort statistics and model training.
2. **Error-flagged rows:** Rows with `Error != 0` had some computation failure; we exclude these for clean analysis.
3. **Ambiguous labels:** Rows labelled `'U'` (uncertain) or `'--'` (unclassified) are removed for supervised classification — only clear labels (0 or 1) are retained.

In [ ]:
SENTINEL = -999  # Any value below this threshold is a sentinel (failed measurement)

# Step 1: Identify rows where ANY feature contains a sentinel value
mask_sentinel = (df[FEATURES] < SENTINEL).any(axis=1)
print(f'Rows with sentinel values: {mask_sentinel.sum():,}')

# Count sentinel values per feature to understand which measurements fail most often
for col in FEATURES:
    n = (df[col] < SENTINEL).sum()
    print(f'  {col}: {n:,} sentinels')

In [ ]:
# Step 2: Identify rows flagged with computation errors
mask_error = df['Error'] != 0
print(f'Rows with Error != 0: {mask_error.sum():,}')

# Step 3: Identify rows with clear, interpretable labels (0 = elliptical, 1 = spiral)
mask_label = df['ML2classes'].isin(['0', '1'])
print(f'Rows with valid binary labels: {mask_label.sum():,}')
print(f'Rows excluded (uncertain/unclassified): {(~mask_label).sum():,}')

In [ ]:
# Apply all three filters simultaneously
df_clean = df[~mask_sentinel & ~mask_error & mask_label].copy()

# Convert label to integer (0 or 1)
df_clean['label'] = df_clean['ML2classes'].astype(int)

# Keep only our working columns and drop any remaining NaN rows
df_clean = df_clean[FEATURES + ['label']].dropna().reset_index(drop=True)

print(f'Original dataset:  {len(df):,} rows')
print(f'Clean dataset:     {len(df_clean):,} rows')
print(f'Removed:           {len(df) - len(df_clean):,} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')
print(f'\nClass distribution:')
print(f'  Elliptical (0): {(df_clean["label"]==0).sum():,}')
print(f'  Spiral     (1): {(df_clean["label"]==1).sum():,}')

**Note on class imbalance:** The dataset is significantly imbalanced — approximately 87% spiral vs 13% elliptical after cleaning. For classification, we apply **stratified balanced sampling** (equal class sizes) to prevent the model from simply predicting the majority class.

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Map numeric labels to readable names for plotting
label_names = {0: 'Elliptical', 1: 'Spiral'}
df_plot = df_clean.copy()
df_plot['Galaxy Type'] = df_plot['label'].map(label_names)

# Plot histograms of each morphological feature, split by galaxy type
# This reveals how well each feature separates the two classes
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Feature Distributions by Galaxy Type', fontsize=15, fontweight='bold')

for ax, feat in zip(axes.flatten(), FEATURES):
    for lbl, color in [('Elliptical', C1), ('Spiral', C2)]:
        grp = df_plot[df_plot['Galaxy Type'] == lbl][feat]
        ax.hist(grp, bins=60, alpha=0.6, label=lbl, density=True, color=color)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation matrix
# Helps identify multicollinearity — highly correlated features may be redundant
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_clean[FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Classification

We compare three supervised classifiers:
- **Decision Tree** — interpretable, mirrors the approach used in the original Barchi et al. paper
- **Random Forest** — ensemble of trees; typically more robust and accurate
- **Logistic Regression** — linear baseline; requires feature scaling

All models are trained on a **balanced stratified sample** (30,000 elliptical + 30,000 spiral), with an 80/20 train-test split.

In [ ]:
# Create a balanced sample — equal numbers of each class — to avoid bias toward spirals
n_per_class = 30000
df_ellip = df_clean[df_clean['label'] == 0].sample(n_per_class, random_state=RANDOM_STATE)
df_spir  = df_clean[df_clean['label'] == 1].sample(n_per_class, random_state=RANDOM_STATE)
df_sample = pd.concat([df_ellip, df_spir]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Balanced sample: {len(df_sample):,} galaxies ({n_per_class:,} per class)')

# Separate features (X) and labels (y)
X = df_sample[FEATURES].values
y = df_sample['label'].values

# Stratified train/test split — preserves class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Training set: {len(X_train):,} | Test set: {len(X_test):,}')

# Scale features for Logistic Regression (tree-based models do not need scaling)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # Fit only on training data
X_test_sc  = scaler.transform(X_test)         # Apply same transform to test data

In [ ]:
# ── Model 1: Decision Tree ──
# max_depth=8 limits tree complexity and prevents overfitting
dt = DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('Decision Tree Results:')
print(classification_report(y_test, y_pred_dt, target_names=['Elliptical', 'Spiral']))
print(f'AUC-ROC: {roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# ── Model 2: Random Forest ──
# 100 trees, each trained on a random feature subset — reduces variance vs single tree
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest Results:')
print(classification_report(y_test, y_pred_rf, target_names=['Elliptical', 'Spiral']))
print(f'AUC-ROC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# ── Model 3: Logistic Regression ──
# Linear model used as a baseline; uses scaled features
lr = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('Logistic Regression Results:')
print(classification_report(y_test, y_pred_lr, target_names=['Elliptical', 'Spiral']))
print(f'AUC-ROC: {roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1]):.4f}')

In [ ]:
# Confusion matrices — visualise where each model makes errors
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices – Classification Models', fontsize=13, fontweight='bold')

model_results = [
    ('Decision Tree',       confusion_matrix(y_test, y_pred_dt)),
    ('Random Forest',       confusion_matrix(y_test, y_pred_rf)),
    ('Logistic Regression', confusion_matrix(y_test, y_pred_lr)),
]
for ax, (name, cm) in zip(axes, model_results):
    ConfusionMatrixDisplay(cm, display_labels=['Elliptical', 'Spiral']).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest — which morphological parameters matter most?
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(len(FEATURES)), importances[idx], color='C1, edgecolor='none')
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels([FEATURES[i] for i in idx], fontsize=11)
ax.set_ylabel('Importance', fontsize=11)
ax.set_title('Random Forest – Feature Importances', fontsize=13, fontweight='bold')
for bar, imp in zip(bars, importances[idx]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{imp:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Summary bar chart comparing all three models on Accuracy and AUC-ROC
model_names  = ['Decision Tree', 'Random Forest', 'Logistic Regression']
accuracies   = [
    (y_pred_dt == y_test).mean(),
    (y_pred_rf == y_test).mean(),
    (y_pred_lr == y_test).mean(),
]
aucs = [
    roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]),
    roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]),
    roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1]),
]

x = np.arange(len(model_names)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, accuracies, w, label='Accuracy', color=C1)
ax.bar(x + w/2, aucs,       w, label='AUC-ROC',  color=C2)
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0.5, 1.0); ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Comparison: Accuracy & AUC-ROC', fontsize=13, fontweight='bold')
ax.legend()
for xi, (a, u) in zip(x, zip(accuracies, aucs)):
    ax.text(xi - w/2, a + 0.005, f'{a:.3f}', ha='center', fontsize=8)
    ax.text(xi + w/2, u + 0.005, f'{u:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Clustering — K-Means

Unsupervised clustering asks whether the morphological parameter space naturally organises into groups, without using labels. This is valuable because:
- It can reveal substructure within broad morphological classes (e.g., compact vs. diffuse spirals)
- It tests whether the data naturally separates in a way consistent with the supervised labels

We use **K-Means** and select the optimal k using both the Elbow Method (inertia) and Silhouette Score.

In [ ]:
# Take a random 20,000-galaxy sample for clustering (K-Means scales quadratically)
df_clust = df_clean.sample(20000, random_state=RANDOM_STATE).copy().reset_index(drop=True)

# Scale features — K-Means is distance-based, so feature scaling is essential
scaler_cl = StandardScaler()
X_cl = scaler_cl.fit_transform(df_clust[FEATURES].values)

print(f'Clustering on {len(df_clust):,} galaxies with {len(FEATURES)} features')

In [ ]:
# Elbow Method + Silhouette Score — test k from 2 to 8
# Inertia: sum of squared distances to nearest centroid (lower = tighter clusters)
# Silhouette: measures how well-separated clusters are (higher = better, max=1)
inertias, sil_scores, K_range = [], [], range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cl)
    inertias.append(km.inertia_)
    # Use a subsample of 5000 for silhouette (computationally expensive at O(n^2))
    sil = silhouette_score(X_cl, km.labels_, sample_size=5000, random_state=RANDOM_STATE)
    sil_scores.append(sil)
    print(f'k={k}: inertia={km.inertia_:.0f}, silhouette={sil:.4f}')

In [ ]:
# Plot the Elbow curve and Silhouette scores to choose optimal k
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, 'o-', color=C1, linewidth=2)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')

axes[1].plot(list(K_range), sil_scores, 'o-', color=C2, linewidth=2)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score by k', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette: k={best_k} (score={max(sil_scores):.4f})')

In [ ]:
# Final K-Means with k=3 — chosen to probe substructure beyond the binary elliptical/spiral split
# (k=2 has the highest silhouette, but k=3 reveals interpretable morphological subgroups)
km_final = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
km_final.fit(X_cl)
df_clust['cluster'] = km_final.labels_

# PCA projection to 2D for visualisation
# PCA finds the directions of maximum variance — ideal for high-dimensional scatter plots
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cl)
df_clust['PC1'] = X_pca[:, 0]
df_clust['PC2'] = X_pca[:, 1]

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Combined: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# Visualise clusters in PCA space and compare morphological profiles
cluster_colors = [C1, C2, C3]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter plot in PCA space coloured by cluster
for c in range(3):
    mask = df_clust['cluster'] == c
    axes[0].scatter(df_clust.loc[mask, 'PC1'], df_clust.loc[mask, 'PC2'],
                    s=2, alpha=0.3, color=cluster_colors[c], label=f'Cluster {c}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=10)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=10)
axes[0].set_title('K-Means Clusters (PCA projection)', fontsize=12, fontweight='bold')
axes[0].legend(markerscale=5)

# Right: radar-style line plot showing the normalised mean feature values per cluster
# This tells us the 'personality' of each cluster
cluster_means = df_clust.groupby('cluster')[FEATURES].mean()
cmn = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())
for i, row in cmn.iterrows():
    axes[1].plot(FEATURES, row.values, 'o-', label=f'Cluster {i}',
                 color=cluster_colors[i], linewidth=2)
axes[1].set_ylabel('Normalised Mean Value', fontsize=10)
axes[1].set_title('Cluster Morphological Profiles', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Validate clusters against known labels
# If clusters align with morphological types, unsupervised learning is capturing real structure
comp = df_clust.groupby('cluster')['label'].value_counts(normalize=True).unstack().fillna(0)
comp.columns = ['Elliptical (0)', 'Spiral (1)']
print('Cluster true-label composition (proportion):')
print(comp.round(3))

print('\nCluster mean feature values:')
print(cluster_means.round(4))

## 7. Summary of Findings

### Classification
| Model | Accuracy | AUC-ROC |
|---|---|---|
| Decision Tree | 96.97% | 0.9912 |
| Random Forest | **97.81%** | **0.9973** |
| Logistic Regression | 90.91% | 0.9741 |

- **Random Forest** achieves the best performance (97.8% accuracy, AUC 0.997), suggesting that an ensemble of decision trees substantially improves upon a single tree with minimal additional cost.
- **Smoothness (S)** and **Concentration (C)** are the most important features for separating elliptical from spiral galaxies.
- **Logistic Regression** reaches 90.9% accuracy, confirming that even a linear boundary captures most of the morphological signal, but non-linear boundaries provide a meaningful boost.

### Clustering
- K-Means with **k=2** achieves the highest silhouette score (0.349), consistent with the two dominant morphological classes.
- With **k=3**, the algorithm reveals an interpretable third group: a population of compact spirals with low Smoothness and low G2 values, distinct from diffuse spirals.
- Cluster 1 (k=3) is strongly enriched in elliptical galaxies (69% elliptical), confirming that the unsupervised approach recovers genuine astrophysical structure.